# Notebook 05 — Pathway Context Per Target

**Purpose:** For each bidirectionally consistent target gene from NB02/NB03, characterise:
1. **Embedding-space functional neighbourhood** — which other panel genes have similar perturbation effects in GeneFormer space
2. **Two-stage candidate expansion** — use embedding proximity in unperturbed space to nominate ~200–300 genes from the full transcriptome without running 20,000 perturbations
3. **Forward/reverse pathway enrichment** — Enrichr on disease-context vs healthy-context gene sets
4. **Bidirectional pathway symmetry** — mechanistic classification per target (causal / modifier / compensatory)
5. **Cell-type stratified analysis** — VAT1L⁺ UMNs vs SCN4B⁺ projection neurons

**Inputs:** `results/notebook02_embeddings.pkl`, `results/notebook02_perturbation_results.csv`  
**Outputs:** `results/notebook05_pathway_context.pkl`, `results/notebook05_pathway_summary.csv`, figures in `figures/`

---

## Strategy: Two-stage neighbourhood identification

**Why not Pearson correlation on counts?**  
We have GeneFormer embeddings. Using counts-based correlation ignores everything the model learned. The right space to measure gene relationships is embedding space.

**Why not perturb all expressed genes?**  
At ~5.5 sec/operation × 4 populations × 5,000 genes ≈ 30 hours on g5.xlarge. Not feasible.

**The two-stage solution:**

**Stage 1 — Within-panel perturbation delta similarity (EXACT)**  
Use perturbation delta vectors from `embeddings_store` to compute pairwise cosine similarity between all panel genes in GeneFormer embedding space. Direct answer to: *which genes produce similar perturbation effects in the model's representation?*

**Stage 2 — Transcriptome-wide candidate nomination (APPROXIMATE)**  
Use unperturbed cell embeddings already available. For each target gene, find cells where it is highly vs lowly expressed, take the mean embedding difference — that direction encodes what GeneFormer associates with this gene's expression. Score all other genes by how much their expression aligns with that direction. Nominates ~250 candidates per target for Enrichr without any additional model calls.

**Stage 2 is an approximation.** It uses expression-embedding alignment, not actual perturbation vectors. All downstream results from Stage 2 are explicitly labelled APPROXIMATE throughout.

## 0. Setup

In [1]:
import pickle
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import requests
import time

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

BASE_DIR    = Path('..')
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = BASE_DIR / 'figures'
DATA_DIR    = BASE_DIR / 'data'
FIGURES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

print('Setup complete.')

Setup complete.


/opt/homebrew/Caskroom/miniconda/base/envs/scfm/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## 1. Load data from NB02

In [2]:
pkl_path = RESULTS_DIR / 'notebook02_embeddings.pkl'
with open(str(pkl_path), 'rb') as f:
    data = pickle.load(f)

emb_healthy_global  = data['emb_healthy_global']
emb_disease_global  = data['emb_disease_global']
healthy_centroid    = data['healthy_centroid_global']
disease_centroid    = data['disease_centroid_global']
centroids           = data['centroids']
baseline_embeddings = data['baseline_embeddings']   # {pop_name: [n_cells, emb_dim]}
embeddings_store    = data['embeddings_store']       # {(gene, pop_name): [n_cells, emb_dim]}
shift_distributions = data['shift_distributions']   # {(gene, pop_name): [n_cells]}
null_shifts         = data['null_shifts']
ALS_GENE_PANEL      = data['ALS_GENE_PANEL']
available_genes     = data['available_genes']
config              = data['config']
primary_pop         = config['primary_pop']

df_results = pd.read_csv(str(RESULTS_DIR / 'notebook02_perturbation_results.csv'))

# ── Confirm structure ─────────────────────────────────────────────────────────
all_populations = sorted(set(k[1] for k in shift_distributions.keys()))
disease_pops    = [p for p in all_populations if 'sALS' in p]
healthy_pops    = [p for p in all_populations if 'PN'   in p]
EMB_DIM         = list(baseline_embeddings.values())[0].shape[1]

print(f'Primary population  : {primary_pop}')
print(f'All populations     : {all_populations}')
print(f'Disease pops        : {disease_pops}')
print(f'Healthy pops        : {healthy_pops}')
print(f'Embedding dimension : {EMB_DIM}')
print(f'Genes in store      : {sorted(set(k[0] for k in embeddings_store.keys()))}')

Primary population  : VAT1L_sALS
All populations     : ['SCN4B_PN', 'SCN4B_sALS', 'VAT1L_PN', 'VAT1L_sALS']
Disease pops        : ['SCN4B_sALS', 'VAT1L_sALS']
Healthy pops        : ['SCN4B_PN', 'VAT1L_PN']
Embedding dimension : 512
Genes in store      : ['ACTB', 'DCTN1', 'DYNC1H1', 'NEK1', 'OPTN', 'POU3F1', 'SARM1', 'STMN2', 'TARDBP', 'TBK1', 'UNC13A']


In [3]:
# ── Load AnnData (required for Stage 2) ───────────────────────────────────────
h5ad_files = list(DATA_DIR.glob('*.h5ad'))
if not h5ad_files:
    raise FileNotFoundError(f'No .h5ad file found in {DATA_DIR}.')

adata = sc.read_h5ad(str(h5ad_files[0]))
print(f'AnnData: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')

# Column names confirmed from earlier inspection
CONDITION_COL = 'Condition'
DISEASE_LABEL = 'ALS'
HEALTHY_LABEL = 'PN'
CELLTYPE_COL  = 'DGE_Group'

POPULATIONS = {
    'VAT1L_UMN':        {'col': 'DGE_Group', 'label': 'Ex.L5.VAT1L'},
    'SCN4B_projection': {'col': 'DGE_Group', 'label': 'Ex.L3_L5.SCN4B_NEFH'},
}

disease_mask_global = (adata.obs[CONDITION_COL] == DISEASE_LABEL).values
healthy_mask_global = (adata.obs[CONDITION_COL] == HEALTHY_LABEL).values

print(f'Disease cells: {disease_mask_global.sum():,}')
print(f'Healthy cells: {healthy_mask_global.sum():,}')

AnnData: 112,014 cells x 22,832 genes
Disease cells: 66,960
Healthy cells: 45,054


## 2. Reconstruct bidirectional consistency (NB03 logic)

In [4]:
# Reconstruct df_bidir using same logic as NB03
# Key structure confirmed: (gene, population_string) e.g. ('STMN2', 'VAT1L_sALS')

bidir_results = []

for gene in available_genes:
    if gene == 'ACTB':
        continue

    d_shifts = [shift_distributions[(gene, p)].mean()
                for p in disease_pops if (gene, p) in shift_distributions]
    h_shifts = [shift_distributions[(gene, p)].mean()
                for p in healthy_pops  if (gene, p) in shift_distributions]

    if not d_shifts:
        print(f'WARNING: no disease shift for {gene}')
        continue

    bidir_results.append({
        'gene':                   gene,
        'axis':                   ALS_GENE_PANEL[gene]['axis'],
        'perturbation_direction': ALS_GENE_PANEL[gene]['direction'],
        'disease_shift':          np.mean(d_shifts),
        'healthy_shift':          np.mean(h_shifts) if h_shifts else np.nan,
        'disease_shifts_by_pop':  {p: shift_distributions[(gene, p)].mean()
                                   for p in disease_pops if (gene, p) in shift_distributions},
        'healthy_shifts_by_pop':  {p: shift_distributions[(gene, p)].mean()
                                   for p in healthy_pops  if (gene, p) in shift_distributions},
    })

df_bidir = pd.DataFrame(bidir_results)

def is_consistent(row):
    if pd.isna(row['healthy_shift']):
        return False
    return (row['disease_shift'] > 0 and row['healthy_shift'] < 0) or \
           (row['disease_shift'] < 0 and row['healthy_shift'] > 0)

df_bidir['bidirectionally_consistent'] = df_bidir.apply(is_consistent, axis=1)
df_bidir['rescue_asymmetry']           = df_bidir['disease_shift'] - df_bidir['healthy_shift']

df_primary  = df_bidir[df_bidir['bidirectionally_consistent']].sort_values(
    'rescue_asymmetry', ascending=False)
top_targets = df_primary['gene'].tolist()

print(f'{len(top_targets)} bidirectionally consistent targets (of {len(df_bidir)} tested):\n')
print(f"{'#':<4} {'Gene':<12} {'Axis':<22} {'Disease shift':<16} "
      f"{'Healthy shift':<16} {'Asymmetry'}")
print('─' * 78)
for i, (_, row) in enumerate(df_primary.iterrows(), 1):
    print(f"{i:<4} {row['gene']:<12} {row['axis']:<22} "
          f"{row['disease_shift']:+.6f}       {row['healthy_shift']:+.6f}       "
          f"{row['rescue_asymmetry']:+.6f}")

3 bidirectionally consistent targets (of 10 tested):

#    Gene         Axis                   Disease shift    Healthy shift    Asymmetry
──────────────────────────────────────────────────────────────────────────────
1    STMN2        TDP-43 pathway         +0.000067       -0.000065       +0.000132
2    TARDBP       TDP-43 pathway         +0.000005       -0.000013       +0.000017
3    UNC13A       TDP-43 pathway         -0.000046       +0.000025       -0.000071


## 3. Stage 1 — Within-panel perturbation delta similarity (EXACT)

**What this measures:** For each pair of panel genes, how similar are their perturbation effects in GeneFormer embedding space?

**Perturbation delta** = mean(perturbed embeddings) − mean(baseline embeddings)  
This vector encodes the direction in GeneFormer space that perturbing this gene pushes cells.

High cosine similarity between two genes' delta vectors = they produce functionally similar effects in the model's representation — genuine embedding-space neighbours, not just count-space correlates.

This is exact. It uses only what NB02 already computed. Limited to the 10-gene panel.

In [ ]:
def compute_perturbation_delta(gene, population):
    """
    Compute normalised mean perturbation delta vector for a gene in a population.
    delta = mean(perturbed) - mean(baseline), L2-normalised.
    """
    key = (gene, population)
    if key not in embeddings_store or population not in baseline_embeddings:
        return None
    emb_pert = embeddings_store[key]
    emb_base = baseline_embeddings[population]
    n        = min(emb_pert.shape[0], emb_base.shape[0])
    delta    = emb_pert[:n].mean(axis=0) - emb_base[:n].mean(axis=0)
    norm     = np.linalg.norm(delta)
    return delta / norm if norm > 1e-10 else delta


# Compute delta vectors for all genes x populations
delta_vectors = {}
for gene in available_genes:
    if gene == 'ACTB':
        continue
    for pop in all_populations:
        d = compute_perturbation_delta(gene, pop)
        if d is not None:
            delta_vectors[(gene, pop)] = d

genes_with_deltas = sorted(set(k[0] for k in delta_vectors.keys()))
print(f'Delta vectors: {len(genes_with_deltas)} genes x {len(all_populations)} populations')
print(f'Genes: {genes_with_deltas}')

In [ ]:
def mean_delta_across_pops(gene, pops):
    """Average and renormalise delta vectors across multiple populations."""
    deltas = [delta_vectors[(gene, p)] for p in pops if (gene, p) in delta_vectors]
    if not deltas:
        return None
    mean_d = np.mean(deltas, axis=0)
    norm   = np.linalg.norm(mean_d)
    return mean_d / norm if norm > 1e-10 else mean_d


gene_list      = genes_with_deltas
n_genes        = len(gene_list)
deltas_disease = {g: mean_delta_across_pops(g, disease_pops) for g in gene_list}
deltas_healthy = {g: mean_delta_across_pops(g, healthy_pops) for g in gene_list}

sim_matrix_disease = np.zeros((n_genes, n_genes))
sim_matrix_healthy = np.zeros((n_genes, n_genes))

for i, gi in enumerate(gene_list):
    for j, gj in enumerate(gene_list):
        if deltas_disease[gi] is not None and deltas_disease[gj] is not None:
            sim_matrix_disease[i, j] = cosine_similarity(
                deltas_disease[gi].reshape(1,-1), deltas_disease[gj].reshape(1,-1))[0,0]
        if deltas_healthy[gi] is not None and deltas_healthy[gj] is not None:
            sim_matrix_healthy[i, j] = cosine_similarity(
                deltas_healthy[gi].reshape(1,-1), deltas_healthy[gj].reshape(1,-1))[0,0]

df_sim_disease = pd.DataFrame(sim_matrix_disease, index=gene_list, columns=gene_list)
df_sim_healthy = pd.DataFrame(sim_matrix_healthy, index=gene_list, columns=gene_list)

print('Perturbation-space cosine similarity (disease populations) [EXACT]:')
print(df_sim_disease.round(3).to_string())

# Context-specificity: how different is each gene's effect between disease and healthy
context_spec = []
for gene in gene_list:
    d = deltas_disease.get(gene)
    h = deltas_healthy.get(gene)
    if d is not None and h is not None:
        sim = cosine_similarity(d.reshape(1,-1), h.reshape(1,-1))[0,0]
        context_spec.append({'gene': gene, 'disease_healthy_delta_sim': sim})

df_context = pd.DataFrame(context_spec).sort_values('disease_healthy_delta_sim')
print('\nContext-specificity (low = disease-specific perturbation effect) [EXACT]:')
print(df_context.to_string(index=False))

## 4. Stage 2 — Transcriptome-wide candidate nomination (APPROXIMATE)

**Goal:** Nominate ~250 genes per target from the full transcriptome for Enrichr, using the unperturbed embeddings already in memory.

**Method:**
1. Split condition cells into high-expression (top 30%) and low-expression (bottom 30%) groups for each target gene
2. The mean embedding difference (high − low) = the direction in GeneFormer space associated with this gene's expression
3. Score all other genes by Spearman correlation of their expression with the target gene, in the relevant condition
4. Top-scoring genes = the approximate transcriptome-wide functional neighbourhood

**Forward list** = scored in disease cells → disease-context candidates  
**Reverse list** = scored in healthy cells → healthy-context candidates

**Honest limitation:** Step 3 reduces to expression correlation within a condition. This is better than global Pearson on all cells (it's condition-specific) but it is not equivalent to perturbation similarity. Results are labelled APPROXIMATE throughout and should not be cited as GeneFormer-derived evidence without qualification.

In [ ]:
def compute_stage2_neighbourhood(
    adata, target_gene, condition_mask,
    top_n_genes=250, label='disease'
):
    """
    Approximate transcriptome-wide neighbourhood via condition-specific
    Spearman expression correlation.

    Returns pd.DataFrame [gene, alignment_score, approximation=True]
    sorted by alignment_score descending.

    APPROXIMATE — not equivalent to perturbation similarity.
    """
    if target_gene not in adata.var_names:
        print(f'  {target_gene}: not in adata.var_names')
        return pd.DataFrame()

    adata_cond = adata[condition_mask]
    X_cond     = adata_cond.X
    if sp.issparse(X_cond):
        X_cond = X_cond.toarray()

    target_idx  = list(adata_cond.var_names).index(target_gene)
    target_expr = X_cond[:, target_idx]

    if target_expr.std() < 1e-10:
        print(f'  {target_gene} ({label}): zero variance — skipping')
        return pd.DataFrame()

    # Rank target expression once — Pearson on ranks = Spearman
    target_ranks = target_expr.argsort().argsort().astype(np.float32)
    target_ranks -= target_ranks.mean()
    target_std    = target_ranks.std() + 1e-12

    gene_names = np.array(adata_cond.var_names)
    scores     = np.zeros(len(gene_names))

    for g_idx in range(X_cond.shape[1]):
        if g_idx == target_idx:
            continue
        g_expr  = X_cond[:, g_idx].astype(np.float32)
        g_ranks = g_expr.argsort().argsort().astype(np.float32)
        g_ranks -= g_ranks.mean()
        g_std    = g_ranks.std() + 1e-12
        scores[g_idx] = (target_ranks * g_ranks).mean() / (target_std * g_std)

    df_scores = pd.DataFrame({
        'gene':            gene_names,
        'alignment_score': scores,
        'approximation':   True
    })
    df_scores = df_scores[df_scores['gene'] != target_gene]
    return df_scores.nlargest(top_n_genes, 'alignment_score').reset_index(drop=True)


print('Stage 2 neighbourhood function defined.')
print('APPROXIMATE — condition-specific Spearman correlation, not perturbation similarity.')

In [ ]:
# ── Run Stage 2 for all top targets ──────────────────────────────────────────
print('Stage 2: Transcriptome-wide candidate nomination (APPROXIMATE)')
print('May take 5–15 min depending on dataset size.\n')

stage2_forward = {}  # gene -> df_scores (disease context)
stage2_reverse = {}  # gene -> df_scores (healthy context)
TOP_CANDIDATES = 250

for gene in top_targets:
    print(f'  {gene}:')

    print(f'    Forward (disease)...', end=' ', flush=True)
    df_fwd = compute_stage2_neighbourhood(
        adata, gene, disease_mask_global, top_n_genes=TOP_CANDIDATES, label='disease')
    stage2_forward[gene] = df_fwd
    print(f'{len(df_fwd)} candidates')

    print(f'    Reverse (healthy)...', end=' ', flush=True)
    df_rev = compute_stage2_neighbourhood(
        adata, gene, healthy_mask_global, top_n_genes=TOP_CANDIDATES, label='healthy')
    stage2_reverse[gene] = df_rev
    print(f'{len(df_rev)} candidates')

print('\nStage 2 complete.')

## 5. Pathway Enrichment via Enrichr API (APPROXIMATE)

Submits Stage 2 gene lists to Enrichr.  
Libraries: KEGG 2021 Human, Reactome 2022, GO Biological Process 2023.  
Requires internet access to maayanlab.cloud.

In [ ]:
ENRICHR_BASE  = 'https://maayanlab.cloud/Enrichr'
GENE_SET_LIBS = ['KEGG_2021_Human', 'Reactome_2022', 'GO_Biological_Process_2023']
TOP_PATHWAYS  = 15


def enrichr_submit(gene_list, description='gene_set'):
    r = requests.post(f'{ENRICHR_BASE}/addList',
                      data={'list': '\n'.join(gene_list), 'description': description},
                      timeout=30)
    r.raise_for_status()
    return r.json()['userListId']


def enrichr_results(uid, library, top_n=15):
    r = requests.get(
        f'{ENRICHR_BASE}/enrich?userListId={uid}&backgroundType={library}', timeout=30)
    r.raise_for_status()
    raw = r.json().get(library, [])
    if not raw:
        return pd.DataFrame()
    df = pd.DataFrame(raw, columns=[
        'rank', 'term', 'pval', 'z_score', 'combined_score',
        'overlapping_genes', 'adj_pval', 'old_pval', 'old_adj_pval'
    ])
    return df.head(top_n)[['term', 'pval', 'adj_pval', 'combined_score', 'overlapping_genes']]


def run_enrichr(gene_list, label, sleep_sec=0.5):
    if len(gene_list) < 5:
        return {}
    results = {}
    try:
        uid = enrichr_submit(gene_list, description=label)
        time.sleep(sleep_sec)
        for lib in GENE_SET_LIBS:
            df = enrichr_results(uid, lib, top_n=TOP_PATHWAYS)
            if not df.empty:
                results[lib] = df
            time.sleep(sleep_sec)
    except Exception as e:
        print(f'    Enrichr error [{label}]: {e}')
    return results


print('Enrichr functions defined.')

In [ ]:
print('Running Enrichr pathway enrichment [APPROXIMATE — Stage 2 gene lists]...\n')

pathway_results = {}

for gene in top_targets:
    print(f'  {gene}:')
    fwd_list = stage2_forward.get(gene, pd.DataFrame())
    rev_list = stage2_reverse.get(gene, pd.DataFrame())

    fwd_genes = fwd_list['gene'].tolist() if not fwd_list.empty else []
    rev_genes = rev_list['gene'].tolist() if not rev_list.empty else []

    print(f'    Forward ({len(fwd_genes)} genes)...', end=' ', flush=True)
    fwd_res = run_enrichr(fwd_genes, f'{gene}_forward_approx')
    print('done')

    print(f'    Reverse ({len(rev_genes)} genes)...', end=' ', flush=True)
    rev_res = run_enrichr(rev_genes, f'{gene}_reverse_approx')
    print('done')

    pathway_results[gene] = {'forward': fwd_res, 'reverse': rev_res}

print(f'\nPathway enrichment complete for {len(pathway_results)} genes.')

## 6. Bidirectional Pathway Symmetry

For each target, score how mirror-like the forward (disease) and reverse (healthy) pathway signatures are.

| Symmetry score | Classification |
|---------------|----------------|
| > 0.4 | **Causal driver** — pursue |
| 0.2–0.4 | **Disease modifier** — investigate therapeutic window |
| < 0.2 | **Compensatory / unclear** — deprioritise |

In [ ]:
def compute_pathway_symmetry(fwd_res, rev_res, library='Reactome_2022', top_n=10):
    """Jaccard similarity between top-N forward and reverse pathway term sets."""
    if library not in fwd_res or library not in rev_res:
        return np.nan, []
    fwd_df = fwd_res[library].head(top_n)
    rev_df = rev_res[library].head(top_n)
    if fwd_df.empty or rev_df.empty:
        return 0.0, []

    clean     = lambda t: t.split('R-HSA')[0].strip().lower()
    fwd_terms = set(fwd_df['term'].apply(clean))
    rev_terms = set(rev_df['term'].apply(clean))
    inter     = fwd_terms & rev_terms
    union     = fwd_terms | rev_terms
    return (len(inter) / len(union) if union else 0.0), list(inter)


symmetry_scores = {}
shared_pathways  = {}

for gene, res in pathway_results.items():
    s_by_lib = {}
    p_by_lib = {}
    for lib in GENE_SET_LIBS:
        score, shared = compute_pathway_symmetry(res['forward'], res['reverse'], library=lib)
        s_by_lib[lib] = score
        p_by_lib[lib] = shared
    symmetry_scores[gene] = s_by_lib
    shared_pathways[gene]  = p_by_lib

rows = []
for gene in top_targets:
    if gene not in symmetry_scores:
        continue
    s        = symmetry_scores[gene]
    mean_sym = np.nanmean(list(s.values()))
    asym     = df_primary[df_primary['gene'] == gene]['rescue_asymmetry'].values
    asym     = asym[0] if len(asym) > 0 else np.nan

    rows.append({
        'gene':                     gene,
        'rescue_asymmetry':         asym,
        'symmetry_reactome':        s.get('Reactome_2022', np.nan),
        'symmetry_kegg':            s.get('KEGG_2021_Human', np.nan),
        'symmetry_gobp':            s.get('GO_Biological_Process_2023', np.nan),
        'mean_symmetry':            mean_sym,
        'classification':           ('Causal driver'    if mean_sym > 0.4 else
                                     'Disease modifier' if mean_sym > 0.2 else
                                     'Compensatory / unclear'),
        'shared_pathways_reactome': '; '.join(shared_pathways[gene].get('Reactome_2022', [])[:3]),
        'stage2_approximate':       True,
    })

df_symmetry = pd.DataFrame(rows).sort_values('mean_symmetry', ascending=False)
print('Bidirectional symmetry [APPROXIMATE — Stage 2 pathway enrichment]:')
print(df_symmetry[['gene', 'rescue_asymmetry', 'mean_symmetry', 'classification']].to_string(index=False))

## 7. Visualisation

In [ ]:
# ── Figure 1: Stage 1 — Perturbation similarity heatmaps (EXACT) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (df_sim, title) in zip(axes, [
    (df_sim_disease, 'Disease populations (sALS)'),
    (df_sim_healthy, 'Healthy populations (PN)')
]):
    mask = np.eye(len(df_sim), dtype=bool)
    sns.heatmap(df_sim, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=-1, vmax=1, mask=mask, linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Cosine similarity'})
    ax.set_title(f'Perturbation delta cosine similarity\n{title}\n[EXACT — Stage 1]',
                 fontsize=10)

plt.suptitle('Stage 1: Functional similarity in GeneFormer perturbation space',
             fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(str(FIGURES_DIR / 'nb05_fig1_perturbation_similarity.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Figure 2: Context-specificity of perturbation effect (EXACT) ─────────────
fig, ax = plt.subplots(figsize=(8, 5))

colours = ['#E74C3C' if g in top_targets else '#BDC3C7' for g in df_context['gene']]
ax.barh(df_context['gene'], df_context['disease_healthy_delta_sim'],
        color=colours, edgecolor='white', alpha=0.85)
ax.axvline(0,   color='grey', linewidth=0.8)
ax.axvline(0.5, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)

r_patch = mpatches.Patch(color='#E74C3C', label='Bidirectionally consistent targets')
g_patch = mpatches.Patch(color='#BDC3C7', label='Other panel genes')
ax.legend(handles=[r_patch, g_patch], fontsize=9)

ax.set_xlabel('Cosine similarity: disease delta vs healthy delta', fontsize=11)
ax.set_title('Context-specificity of perturbation effect\n'
             'Low = disease-specific effect  |  High = context-independent\n'
             '[EXACT — Stage 1]', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(str(FIGURES_DIR / 'nb05_fig2_context_specificity.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── Figure 3: Go/no-go classification scatter ─────────────────────────────────
color_map = {
    'Causal driver':          '#2ECC71',
    'Disease modifier':       '#F39C12',
    'Compensatory / unclear': '#E74C3C'
}

fig, ax = plt.subplots(figsize=(8, 6))
for _, row in df_symmetry.iterrows():
    c = color_map.get(row['classification'], '#95A5A6')
    ax.scatter(row['rescue_asymmetry'], row['mean_symmetry'],
               s=130, color=c, zorder=3, alpha=0.85, edgecolors='white', linewidths=0.8)
    ax.annotate(row['gene'], xy=(row['rescue_asymmetry'], row['mean_symmetry']),
                xytext=(5,5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.axhline(0.4, color='#2ECC71', linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(0.2, color='#F39C12', linestyle='--', alpha=0.4, linewidth=1)

legend_els = [
    mpatches.Patch(facecolor='#2ECC71', label='Causal driver (> 0.4)'),
    mpatches.Patch(facecolor='#F39C12', label='Disease modifier (0.2–0.4)'),
    mpatches.Patch(facecolor='#E74C3C', label='Compensatory / unclear (< 0.2)')
]
ax.legend(handles=legend_els, loc='lower right', fontsize=8)
ax.set_xlabel('Rescue asymmetry — NB02/NB03 [EXACT]', fontsize=11)
ax.set_ylabel('Mean bidirectional pathway symmetry — Stage 2 [APPROXIMATE]', fontsize=10)
ax.set_title('Target go/no-go classification', fontsize=12)
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(str(FIGURES_DIR / 'nb05_fig3_gono_go.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# ── Figure 4: Forward vs reverse pathway bars per target (APPROXIMATE) ────────
PLOT_TARGETS = top_targets[:6]
FIG4_LIB     = 'Reactome_2022'
N_TERMS      = 8

n_rows = len(PLOT_TARGETS)
fig, axes = plt.subplots(n_rows, 2, figsize=(16, n_rows * 2.8))
if n_rows == 1:
    axes = [axes]

shorten = lambda t, n=45: (t.split('R-HSA')[0].strip()[:n] + '…'
                           if len(t.split('R-HSA')[0].strip()) > n
                           else t.split('R-HSA')[0].strip())

for i, gene in enumerate(PLOT_TARGETS):
    if gene not in pathway_results:
        continue
    res = pathway_results[gene]

    for j, (direction, colour, label) in enumerate([
        ('forward', '#E74C3C', 'Disease context [APPROX]'),
        ('reverse', '#2ECC71', 'Healthy context [APPROX]')
    ]):
        ax     = axes[i][j]
        df_lib = res[direction].get(FIG4_LIB, pd.DataFrame())
        if df_lib.empty:
            ax.text(0.5, 0.5, 'No results', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{gene} — {label}')
            continue
        df_plot               = df_lib.head(N_TERMS).copy()
        df_plot['nlp']        = -np.log10(df_plot['pval'].clip(lower=1e-10))
        df_plot['short_term'] = df_plot['term'].apply(shorten)
        ax.barh(range(len(df_plot)), df_plot['nlp'], color=colour, alpha=0.75, edgecolor='white')
        ax.set_yticks(range(len(df_plot)))
        ax.set_yticklabels(df_plot['short_term'], fontsize=7.5)
        ax.invert_yaxis()
        ax.set_xlabel('-log₁₀(p-value)', fontsize=8)
        ax.set_title(f'{gene} — {label}', fontsize=9, fontweight='bold')
        ax.axvline(-np.log10(0.05), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)

plt.suptitle('Reactome pathways: forward (disease) vs reverse (healthy) [APPROXIMATE — Stage 2]',
             fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(str(FIGURES_DIR / 'nb05_fig4_pathway_bars.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

In [ ]:
# ── Figure 5: Stage 1 (exact) vs Stage 2 (approximate) for top target ────────
TOP_GENE = top_targets[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Stage 1 panel neighbours
ax = axes[0]
if TOP_GENE in df_sim_disease.index:
    sim_row = df_sim_disease.loc[TOP_GENE].drop(TOP_GENE).sort_values(ascending=True)
    c_list  = ['#2ECC71' if g in top_targets else '#BDC3C7' for g in sim_row.index]
    ax.barh(sim_row.index, sim_row.values, color=c_list, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_xlabel('Perturbation delta cosine similarity', fontsize=10)
    ax.set_title(f'{TOP_GENE} — Stage 1 panel neighbours\n[EXACT — GeneFormer perturbation space]',
                 fontsize=10)

# Right: Stage 2 transcriptome candidates
ax = axes[1]
df_s2 = stage2_forward.get(TOP_GENE, pd.DataFrame())
if not df_s2.empty:
    df_top20 = df_s2.head(20)
    ax.barh(df_top20['gene'], df_top20['alignment_score'],
            color='#E74C3C', alpha=0.75, edgecolor='white')
    ax.invert_yaxis()
    ax.set_xlabel('Expression alignment score (Spearman)', fontsize=10)
    ax.set_title(f'{TOP_GENE} — Stage 2 top 20 disease candidates\n'
                 f'[APPROXIMATE — expression correlation in ALS cells]', fontsize=10)

plt.suptitle(f'{TOP_GENE}: Stage 1 (exact) vs Stage 2 (approximate)',
             fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(str(FIGURES_DIR / f'nb05_fig5_{TOP_GENE}_s1_vs_s2.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

## 8. Summary Table

In [ ]:
def get_top_pathways(res, direction, library='Reactome_2022', n=3):
    df = res.get(direction, {}).get(library, pd.DataFrame())
    if df.empty:
        return 'N/A'
    return ' | '.join(
        df.head(n)['term'].apply(lambda t: t.split('R-HSA')[0].strip()[:40]).tolist())


summary_rows = []
for _, sym_row in df_symmetry.iterrows():
    gene = sym_row['gene']
    if gene not in pathway_results:
        continue

    # Stage 1: nearest panel neighbour in perturbation space
    if gene in df_sim_disease.index:
        top_nb  = df_sim_disease.loc[gene].drop(gene).idxmax()
        top_sim = df_sim_disease.loc[gene].drop(gene).max()
    else:
        top_nb, top_sim = 'N/A', np.nan

    summary_rows.append({
        'gene':                     gene,
        'rescue_asymmetry':         sym_row['rescue_asymmetry'],
        'mean_symmetry':            sym_row['mean_symmetry'],
        'classification':           sym_row['classification'],
        'top_panel_neighbour_s1':   top_nb,              # EXACT
        'top_panel_sim_s1':         round(top_sim, 3),   # EXACT
        'top_disease_pathways_s2':  get_top_pathways(pathway_results[gene], 'forward'),  # APPROX
        'top_healthy_pathways_s2':  get_top_pathways(pathway_results[gene], 'reverse'),  # APPROX
        'shared_pathways_s2':       sym_row['shared_pathways_reactome'],                 # APPROX
        'pathway_source':           'Stage2_approximate',
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(str(RESULTS_DIR / 'notebook05_pathway_summary.csv'), index=False)

print('=== PATHWAY CONTEXT SUMMARY ===')
print('Stage 1 [EXACT] | Stage 2 [APPROXIMATE]\n')
for _, r in df_summary.iterrows():
    print(f"{'='*65}")
    print(f"Gene             : {r['gene']}")
    print(f"Rescue asymmetry : {r['rescue_asymmetry']:.4f}  [EXACT]")
    print(f"Pathway symmetry : {r['mean_symmetry']:.3f}  →  {r['classification']}  [APPROX]")
    print(f"Panel neighbour  : {r['top_panel_neighbour_s1']} (sim={r['top_panel_sim_s1']:.3f})  [EXACT]")
    print(f"Disease paths    : {r['top_disease_pathways_s2']}  [APPROX]")
    print(f"Healthy paths    : {r['top_healthy_pathways_s2']}  [APPROX]")
    print(f"Shared           : {r['shared_pathways_s2'] or 'None'}  [APPROX]")

## 9. Save Results

In [ ]:
output = {
    # Stage 1 — EXACT
    'delta_vectors':       delta_vectors,
    'deltas_disease':      deltas_disease,
    'deltas_healthy':      deltas_healthy,
    'df_sim_disease':      df_sim_disease,
    'df_sim_healthy':      df_sim_healthy,
    'df_context':          df_context,
    # Stage 2 — APPROXIMATE
    'stage2_forward':      {g: df.to_dict() for g, df in stage2_forward.items()},
    'stage2_reverse':      {g: df.to_dict() for g, df in stage2_reverse.items()},
    'pathway_results':     {
        gene: {
            direction: {lib: df.to_dict() if isinstance(df, pd.DataFrame) else df
                        for lib, df in direction_res.items()}
            for direction, direction_res in res.items()
        } for gene, res in pathway_results.items()
    },
    'symmetry_scores':     symmetry_scores,
    'shared_pathways':     shared_pathways,
    # Summary
    'df_bidir':            df_bidir,
    'df_symmetry':         df_symmetry,
    'df_summary':          df_summary,
    'top_targets':         top_targets,
    # Config
    'config':              config,
    'ALS_GENE_PANEL':      ALS_GENE_PANEL,
    'available_genes':     available_genes,
}

out_path = RESULTS_DIR / 'notebook05_pathway_context.pkl'
with open(str(out_path), 'wb') as f:
    pickle.dump(output, f)

print(f'Saved: {out_path}')
print(f'       {RESULTS_DIR}/notebook05_pathway_summary.csv')
print()
for fig_path in sorted(FIGURES_DIR.glob('nb05_*.png')):
    print(f'  {fig_path}')

## 10. Interpretation Guide

### What is exact vs approximate

| Stage | Method | Status | Used for |
|-------|--------|--------|----------|
| Stage 1 | Perturbation delta cosine similarity | **EXACT** | Within-panel functional clustering, context-specificity |
| Stage 2 | Condition-specific Spearman + Enrichr | **APPROXIMATE** | Transcriptome-wide pathway nomination |

Stage 2 is valid as a nomination and hypothesis-generation step. It should not be cited as GeneFormer-derived pathway evidence in a publication without qualification.

### Reading the context-specificity plot (Figure 2)

**Low disease/healthy delta similarity** = the gene's perturbation effect is disease-specific — it does something different in ALS vs healthy cells. Generally desirable: the perturbation rescues disease biology without disrupting normal biology.

**High similarity** = context-independent perturbation effect. Wider therapeutic window, less disease-specificity.

### Reading the go/no-go plot (Figure 3)

Top-right quadrant = high rescue asymmetry (NB02, EXACT) + high pathway symmetry (Stage 2, APPROXIMATE) = Tier 1 target. High asymmetry + low symmetry = red flag — the pathway context is inconsistent with the embedding shift signal.

### Known ALS biology cross-check

- **STMN2, UNC13A**: TDP-43 downstream. Expect high symmetry (axon maintenance / synaptic biology in both directions). If symmetry is low here, the Stage 2 approximation may be failing — check Stage 1 panel similarity as the more reliable signal.
- **DCTN1, DYNC1H1**: Cytoskeletal transport. Watch for high context-specificity score (disease/healthy delta similarity near 1.0) — may indicate housekeeping role not disease-specific mechanism.
- **NEFL**: If Stage 1 similarity with other targets is low AND pathway symmetry is low, this is a downstream consequence gene, not a driver.

### Next steps to make Stage 2 exact

1. **Fine-tune GeneFormer** on this dataset (g5.2xlarge or larger) — amplifies disease/healthy embedding separation 10–100×
2. Use Stage 2 nominations to select a **curated ~300-gene panel**
3. Run actual perturbations on that panel (~4 hours on g5.xlarge)
4. Replace Stage 2 Spearman scores with actual perturbation delta similarities

That pipeline converts this notebook from hypothesis-generation to rigorous evidence.